# Import data from UCI Machine Learning Repository
### Dataset: Online Retail
https://archive.ics.uci.edu/dataset/352/online+retail

In [73]:
pip install ucimlrepo

In [74]:
from ucimlrepo import fetch_ucirepo 
import duckdb
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
df = online_retail.data.original 

In [75]:
df.shape

(541909, 8)

# Data Cleaning
Using SQL to clean data to demonstrate capability

In [ ]:
# Filtering dataset
df = duckdb.sql(
    """
    WITH main_unfiltered AS(
        SELECT
            Description                                                                             AS description,
            Quantity                                                                                AS quantity,
            CAST(STRPTIME(InvoiceDate, '%m/%d/%Y %H:%M') AS DATE)                                   AS invoice_date,
            UnitPrice                                                                               AS unit_price,
            CAST(CustomerID AS STRING)                                                              AS customer_id,
            Country                                                                                 AS country,
            InvoiceNo                                                                               AS invoice_number,
            MAX(CAST(STRPTIME(InvoiceDate, '%m/%d/%Y %H:%M') AS DATE)) OVER ()                      AS global_max_date
        FROM
            df
    ),

    main AS (
        SELECT
            *
        FROM
            main_unfiltered
        WHERE
            invoice_date < DATE_ADD(global_max_date, INTERVAL '-3 MONTH') -- Filter out the last 90 days, this will be used for target variables
            AND description = UPPER(description) -- Filtering to just Products, Products are capitalised and adjustments such as discounts are lowercase
            AND quantity >= 1
            AND unit_price >= 0.01
    ),

    max_invoice_date AS (
        SELECT
            MAX(invoice_date) AS max_date
        FROM 
            main
    ),

    customer_lifespan AS (
        SELECT
            customer_id,
            DATEDIFF('day', MIN(invoice_date), MAX(invoice_date)) AS customer_lifespan_days,
            COUNT(DISTINCT invoice_number) AS life_time_orders,
            SUM(quantity*unit_price) AS total_spend, 
            MAX(invoice_date) AS last_order_date 
        FROM
            main
        WHERE
            customer_id IS NOT NULL
        GROUP BY
            customer_id
    ),

    discount AS (
        SELECT
            customer_id,
            invoice_date,
            SUM(quantity*unit_price) AS discount,
            COUNT(invoice_number) AS number_of_discounts 
        FROM
            main_unfiltered
        WHERE
            description = 'Discount'
            AND invoice_date < DATE_ADD(global_max_date, INTERVAL '-3 MONTH')
        GROUP BY
            customer_id, 
            invoice_date
    ),

    target_variable AS (
        SELECT DISTINCT
            customer_id,
            0 as customer_churned
        FROM 
            main_unfiltered
        WHERE
            invoice_date >= DATE_ADD(global_max_date, INTERVAL '-3 MONTH') -- Filter to the last 90 days of data
    ),
    
    final AS (
        SELECT 
            a.customer_id,
            SUM(a.quantity * a.unit_price)                                              AS total_spend,
            MAX(b.customer_lifespan_days)                                               AS customer_age_days,
            MAX(b.life_time_orders)                                                     AS total_orders,
            SUM(a.quantity * a.unit_price) / MAX(b.life_time_orders)                    AS average_order_value,
            MAX(b.customer_lifespan_days) / (MAX(b.life_time_orders) -1)                AS order_frequency,
            SUM(IFNULL(c.discount * -1, 0))                                             AS total_discount_value,
            MAX(IFNULL(c.number_of_discounts, 0))                                       AS orders_with_discounts,
            DATEDIFF('day', MAX(a.invoice_date), MAX(e.max_date))                       AS days_since_last_order,
            MAX(IFNULL(d.customer_churned, 1))                                          AS customer_churned
        FROM 
            main a
        LEFT JOIN
            customer_lifespan b USING (customer_id)
        LEFT JOIN
            discount c USING (customer_id) -- Can sum create the count of discounts per customer then get the sum for total discount and avg discount
        LEFT JOIN
            target_variable d USING (customer_id)
        CROSS JOIN 
            max_invoice_date e
        GROUP BY
            customer_id
    )

    SELECT * FROM final
    """
).df()

df

,customer_id,total_spend,customer_age_days,total_orders,average_order_value,order_frequency,total_discount_value,orders_with_discounts,days_since_last_order,customer_churned
0,14092.0,2720.22,259,10,272.022000,28.777778,0.0,0,7,0
1,16839.0,10869.28,260,22,494.058182,12.380952,0.0,0,17,0
2,15465.0,5992.13,231,15,399.475333,16.500000,0.0,0,46,0
3,12921.0,10083.47,275,24,420.144583,11.956522,0.0,0,6,0
4,17027.0,655.88,138,4,163.970000,46.000000,0.0,0,87,0
...,...,...,...,...,...,...,...,...,...,...
3355,13867.0,138.85,0,1,138.850000,NaN,0.0,0,126,0
3356,17624.0,400.30,0,1,400.300000,NaN,0.0,0,126,0
3357,14340.0,134.70,0,1,134.700000,NaN,0.0,0,126,1
3358,13711.0,252.06,0,2,126.030000,0.000000,0.0,0,126,1


In [ ]:
df['order_frequency'] = df['order_frequency'].fillna(0)

C:\Users\josef\AppData\Local\Temp\ipykernel_36700\1758294310.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['order_frequency'].fillna(0, inplace=True)


,customer_id,total_spend,customer_age_days,total_orders,average_order_value,order_frequency,total_discount_value,orders_with_discounts,days_since_last_order,customer_churned
0,14092.0,2720.22,259,10,272.022000,28.777778,0.0,0,7,0
1,16839.0,10869.28,260,22,494.058182,12.380952,0.0,0,17,0
2,15465.0,5992.13,231,15,399.475333,16.500000,0.0,0,46,0
3,12921.0,10083.47,275,24,420.144583,11.956522,0.0,0,6,0
4,17027.0,655.88,138,4,163.970000,46.000000,0.0,0,87,0
...,...,...,...,...,...,...,...,...,...,...
3355,13867.0,138.85,0,1,138.850000,0.000000,0.0,0,126,0
3356,17624.0,400.30,0,1,400.300000,0.000000,0.0,0,126,0
3357,14340.0,134.70,0,1,134.700000,0.000000,0.0,0,126,1
3358,13711.0,252.06,0,2,126.030000,0.000000,0.0,0,126,1
